In [ ]:
# build simple agent to understand agentic process
# syntax
'''While goal not done:
      think()
      act()
      observe()
      update_memory()
      ....'''

'While goal not done:\n      think()\n      act()\n      observe()\n      update_memory()\n      ....'

1. Now define simple agent which understand user query, and plans it then makes a tool call and returns result
- Query check math opertions requested, extract numbers then call math function by passing numbers then return the result to the user

In [ ]:
#simple math agent
def simple_math_agent(goal):
  print("Goal is ",goal)
  steps = 0
  memory=[]
  while True:
    if "square" in goal:
      number=int(goal.split("square of ")[1].split()[0])
      memory.append(f"found the number {number}")
      result = number*number
      memory.append(f"completed the result {result}")
      print("agent memory",memory)
      print("final answer ",result)
      break
    else:
      print("This operation is not found")
      break
    steps+=1


In [ ]:
prompt = 'find the square of 5'
simple_math_agent(prompt)

Goal is  find the square of 5
agent memory ['found the number 5', 'completed the result 25']
final answer  25


In [ ]:
import sympy
import torch
import transformers

print("SymPy:", sympy.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)

SymPy: 1.14.0
PyTorch: 2.11.0+cu128
Transformers: 5.16.1


In [ ]:
#adding LLM as an agent for understanding the user query and calling the tool
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def simple_math_agent_llm(goal):
  print(" goal is : ",goal)
  prompt=f"""Exact just the number to square from the give text:"{goal}"...
  Reply with only the number , or "none" if there are no numbers"""
  message=[{"role":"user","content":prompt}]
  text=tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
  inputs=tokenizer(text,return_tensors="pt")
  inputs=inputs.to(model.device)
  output=model.generate(**inputs, max_new_tokens=10, do_sample=True)
  result=tokenizer.decode(output[0])
  print("result",result)

  #compute operation
  if result.isdigit():
    output = result*result
    print("square of given number is : ",output)
  else:
    print("operation not found")


In [ ]:
prompt = 'find the square of 10'
simple_math_agent_llm(prompt)

 goal is :  find the square of 10
result <|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Exact just the number to square from the give text:"find the square of 10"...
  Reply with only the number , or "none" if there are no numbers<|im_end|>
<|im_start|>assistant
100<|im_end|>
operation not found


# A simple Travel Agent
where you can ask about
- city information a tool  
- get weather info
- budget estimation
- currency conversion
- select tool based on user query
- tool execution
- Tool result passesd to LLM and then final answer
- test

In [ ]:
#adding LLM as an agent for understanding the user query and calling the tool
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
model_name = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype="auto")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Define the tools


In [ ]:
from langchain_core.tools import tool

In [ ]:
# 1. Weather tool
@tool
def weather (City:str):
  """Returns the current weather for a given city."""
  weather_data = {'Bangalore':'30C,Cloudy','Udaipur':'35C,Normal','Mumbai':'25C,Rainy'}
  return weather_data.get(City, 'Weather data not found')

In [ ]:
# City Info tool
@tool
def City (City:str):
  """Returns information about a given city."""
  city_info_data = {
      'Bangalore':'Bangalore is Tech Hub of India, lot of hotels and shopping Malls are available',
      'Udaipur':'City of lakes , Lake palace is popular tourist destination',
      'Mumbai' :'Mumbai City of dreams for film makers, artists and actors'
  }
  return city_info_data.get(City, 'city info data not found')

In [ ]:
# Currency Exchange tool
@tool
def Currency(from_currency: str, to_currency: str, amount: float):
    """Convert an amount from one currency to another."""

    rates = {
        "USD_INR": 88.0,
        "EUR_INR": 103.0,
        "GBP_INR": 118.0,
        "INR_USD": 0.0114,
        "INR_EUR": 0.0097,
        "INR_GBP": 0.0085
    }

    key = f"{from_currency.upper()}_{to_currency.upper()}"

    if key not in rates:
        return "Exchange rate not available"

    converted_amount = amount * rates[key]

    return f"{amount} {from_currency.upper()} = {converted_amount:.2f} {to_currency.upper()}"

In [ ]:
import requests

@tool
def OpenWeatherTool(City: str):
    """Returns the current weather for a given city using OpenWeatherMap API.
    Requires a City name as input.
    """
    api_key = "YOUR_OPENWEATHER_API_KEY"
    base_url = "http://api.openweathermap.org/data/2.5/weather?"
    complete_url = f"{base_url}appid={api_key}&q={City}&units=metric"

    try:
        response = requests.get(complete_url)
        response.raise_for_status() # Raise an HTTPError for bad responses (4xx or 5xx)
        weather_data = response.json()

        if weather_data["cod"] != "404":
            main = weather_data["main"]
            weather_description = weather_data["weather"][0]["description"]
            temperature = main["temp"]
            humidity = main["humidity"]
            return f"Weather in {City}: {weather_description}, Temperature: {temperature}°C, Humidity: {humidity}%"
        else:
            return f"City '{City}' not found."
    except requests.exceptions.RequestException as e:
        return f"Error fetching weather for {City}: {e}"

In [ ]:
# store all the tools in a list
tools = [weather , City , Currency]

In [ ]:
# Cross check an example
print(weather.invoke({'City': 'Bangalore'}))
print(City.invoke({'City': 'Bangalore'}))
print(Currency.invoke({'from_currency': 'INR', 'to_currency': 'USD', 'amount': 100}))

30C,Cloudy
Bangalore is Tech Hub of India, lot of hotels and shopping Malls are available
100.0 INR = 1.14 USD


In [ ]:
# tool store to dictionary
tools = {'weather': weather, 'City': City, 'Currency': Currency}

In [ ]:
# Update the tools dictionary to include the new OpenWeatherTool
tools['OpenWeatherTool'] = OpenWeatherTool
print("Updated tools dictionary with OpenWeatherTool.")
# You can test it by running:
# print(run_travel_agent('What is the weather in London?'))

Updated tools dictionary with OpenWeatherTool.


In [ ]:
# now create prompt instuction tokenize, extract city and intent from the prompt
def ask_model(prompt):
  instruction='''you are a tarvel agent
  Available tools:
  weather(city)
  City(city)
  Currency(from_currency,to_currency,amount)
  If a tool is required , return ONLY JSON,
  Example:
  [{"tool":"weather", args:{"city:"Bangalore"}},
  {"tool": Cityinfo, "args":{"city":"Udaipur"}}]

  if no tool is required return empty list
  '''
  message=[{'role': 'system', 'content': instruction},{'role': 'user','content': prompt}]
    # apply tokenizer
  text=tokenizer.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
  inputs=tokenizer(text,return_tensors="pt")
  inputs=inputs.to(model.device)
  output=model.generate(**inputs, max_new_tokens=200, do_sample=True)
  # get full results
  full_result=tokenizer.decode(output[0])
  print("Full model result:", full_result) # Added for debugging

  # Extract only the assistant's JSON response from the full_result
  assistant_start_tag = "<|im_start|>assistant\n"
  assistant_end_tag = "<|im_end|>"

  # Find the *last* occurrence of assistant_start_tag to get the most recent response
  start_index = full_result.rfind(assistant_start_tag)
  end_index = full_result.rfind(assistant_end_tag)

  if start_index != -1 and end_index != -1 and start_index < end_index:
      # Adjust start_index to point after the tag
      json_str = full_result[start_index + len(assistant_start_tag) : end_index].strip()
      print("Extracted JSON from assistant:", json_str) # Added for debugging
      return json_str
  else:
      # If extraction fails, return the full result for debugging or further parsing
      print("Could not reliably extract JSON from model output. Returning full result.")
      return full_result


In [ ]:
# now we need to add LLM , and retrive
# to get result in JSON format define a function to search for tool and return results in json
import json
import re
def parse_tool_calls(model_reply):
  # First, try to extract the last assistant's block if model_reply contains chat template
  assistant_start_tag = "<|im_start|>assistant\n"
  assistant_end_tag = "<|im_end|>"

  start_index_assistant = model_reply.rfind(assistant_start_tag)
  end_index_assistant = model_reply.rfind(assistant_end_tag)

  if start_index_assistant != -1 and end_index_assistant != -1 and start_index_assistant < end_index_assistant:
      # Extract only the content within the last assistant block
      potential_json_str = model_reply[start_index_assistant + len(assistant_start_tag) : end_index_assistant].strip()
  else:
      # If no clear assistant block, assume the whole reply *might* be JSON (e.g., if ask_model already cleaned it)
      potential_json_str = model_reply

  # Now, find the actual JSON array within this potential_json_str
  # Using a non-greedy match for a JSON array, and take the last one found.
  all_matches = list(re.finditer(r"\[.*?\]", potential_json_str, re.DOTALL))

  if not all_matches:
    return []

  # Take the last matched JSON-like string
  json_candidate = all_matches[-1].group(0)

  try:
    calls = json.loads(json_candidate)
  except json.JSONDecodeError:
    return []

  # Ensure it's a list and filter for actual tool calls (dict with 'tool' key)
  if isinstance(calls, list):
      return [c for c in calls if isinstance(c, dict) and c.get('tool')]
  else:
      return []


In [ ]:
# Actions : running the tools requested and collecting the results
def execute_tool (tool_calls):
  results =[]
  for call in tool_calls:
    original_tool_name_from_model = call.get('tool', '') # e.g. 'City' or 'Weather'
    tool_name_for_lookup = original_tool_name_from_model.lower() # e.g. 'city' or 'weather'
    args_from_model = call.get('args', {}) # e.g. {'city': 'udaipur'}

    print(f"DEBUG: execute_tool received call: {call}")
    print(f"DEBUG:   original_tool_name_from_model: {original_tool_name_from_model}")
    print(f"DEBUG:   tool_name_for_lookup: {tool_name_for_lookup}")
    print(f"DEBUG:   args_from_model: {args_from_model}")

    actual_tool_obj = None
    # Find the actual tool object from the 'tools' dict, case-insensitively
    for k, v in tools.items():
        if k.lower() == tool_name_for_lookup:
            actual_tool_obj = v
            break

    if actual_tool_obj:
        try:
            # Adapt args keys to match the tool's expected parameter names
            adapted_args = {}
            if tool_name_for_lookup == 'city' or tool_name_for_lookup == 'weather':
                # These tools expect 'City' as parameter name, model gives 'city' key.
                if 'city' in args_from_model:
                    adapted_args['City'] = args_from_model['city']
                else:
                    # Fallback if 'city' key is missing, might still fail tool validation
                    adapted_args = args_from_model
            elif tool_name_for_lookup == 'currency':
                # Currency tool expects 'from_currency', 'to_currency', 'amount'.
                # Model output seems to match these directly.
                adapted_args = args_from_model
            else:
                # For any other tool, pass args as-is, hoping they match.
                adapted_args = args_from_model

            print(f"DEBUG:   Invoking {original_tool_name_from_model} with adapted_args: {adapted_args}")
            tool_execution_result = actual_tool_obj.invoke(adapted_args)
        except Exception as e:
            tool_execution_result = f"error calling the {original_tool_name_from_model}: {e}"
    else:
        tool_execution_result = f"unknown tool {original_tool_name_from_model}"

    # Append results using the original name and args for clarity, but the tool_execution_result
    results.append({'tool': original_tool_name_from_model, 'args': args_from_model, 'result': tool_execution_result})
  return results


In [ ]:
# now create full agent loop using llm and above functions
def run_travel_agent(prompt):
  print("user prompt:",prompt)
  reply = ask_model(prompt) # pass the prompt to understand goal, tool args
  print("reply = ", reply)
  tool_calls= parse_tool_calls(reply) # pass the reply to tool call , to decide

  if not tool_calls:
    print("No tool calls detected, returning initial reply.")
    return reply

  tool_results = execute_tool(tool_calls) # main tool call happens here (corrected 'execute_tools' to 'execute_tool')
  print ("tool results = ",tool_results)

  # Corrected follow_up to be a single string and moved inside the function
  follow_up_prompt = f"Tool results  :{tool_results}\nusing these results, answer the original question: {prompt}"
  final_ans= ask_model(follow_up_prompt)
  print("Final Answer:", final_ans)
  return final_ans


In [ ]:
run_travel_agent('tell me about udaipur')
#

user prompt: tell me about udaipur
Full model result: <|im_start|>system
you are a tarvel agent
  Available tools:
  weather(city)
  City(city)
  Currency(from_currency,to_currency,amount)
  If a tool is required , return ONLY JSON,
  Example:
  [{"tool":"weather", args:{"city:"Bangalore"}},
  {"tool": Cityinfo, "args":{"city":"Udaipur"}}]

  if no tool is required return empty list
  <|im_end|>
<|im_start|>user
tell me about udaipur<|im_end|>
<|im_start|>assistant
[{"tool":"City","args":{"city":"Udaipur"}}, {}]<|im_end|>
Extracted JSON from assistant: [{"tool":"City","args":{"city":"Udaipur"}}, {}]
reply =  [{"tool":"City","args":{"city":"Udaipur"}}, {}]
DEBUG: execute_tool received call: {'tool': 'City', 'args': {'city': 'Udaipur'}}
DEBUG:   original_tool_name_from_model: City
DEBUG:   tool_name_for_lookup: city
DEBUG:   args_from_model: {'city': 'Udaipur'}
DEBUG:   Invoking City with adapted_args: {'City': 'Udaipur'}
tool results =  [{'tool': 'City', 'args': {'city': 'Udaipur'}, 're

"['The City of Lakes, Lake Palace is a popular tourist destination.']"

In [ ]:
run_travel_agent("tell me about udaipur and convert 100 USD to INR")


user prompt: tell me about udaipur and convert 100 USD to INR
Full model result: <|im_start|>system
you are a tarvel agent
  Available tools:
  weather(city)
  City(city)
  Currency(from_currency,to_currency,amount)
  If a tool is required , return ONLY JSON,
  Example:
  [{"tool":"weather", args:{"city:"Bangalore"}},
  {"tool": Cityinfo, "args":{"city":"Udaipur"}}]

  if no tool is required return empty list
  <|im_end|>
<|im_start|>user
tell me about udaipur and convert 100 USD to INR<|im_end|>
<|im_start|>assistant
[
  {
    "tool": "City",
    "args": { "city": "Udaipur" }
  },
  {
    "tool": "Currency",
    "args": { "from_currency": "USD", "to_currency": "INR", "amount": 100 }
  }
]<|im_end|>
Extracted JSON from assistant: [
  {
    "tool": "City",
    "args": { "city": "Udaipur" }
  },
  {
    "tool": "Currency",
    "args": { "from_currency": "USD", "to_currency": "INR", "amount": 100 }
  }
]
reply =  [
  {
    "tool": "City",
    "args": { "city": "Udaipur" }
  },
  {
    "to

'[\n  {\n    "tool": "City",\n    "args": {\n      "city": "Udaipur"\n    },\n    "result": "City of lakes, Lake palace is popular tourist destination"\n  },\n  {\n    "tool": "Currency",\n    "args": {\n      "from_currency": "USD",\n      "to_currency": "INR",\n      "amount": 100\n    },\n    "result": "100.0 USD = 8800.00 INR"\n  }\n]'